In [65]:
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings('ignore')
import os
from tqdm import tqdm
import sys
import time
import json
import pickle
import random
import re

import spikeinterface.extractors as se
import spikeinterface.preprocessing as spre

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
from sklearn.decomposition import PCA
from scipy.stats import pearsonr
from scipy.interpolate import interp1d
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import umap
from umap import UMAP


In [66]:
spike_inf = pd.read_csv("/media/ubuntu/sda/Spike_Sorting/paper_architecture/01_real_data/01_flexible_probe_30_channels/kilosort_spike_sorting/sorting_new/021322/spike_inf.tsv", sep = '\t', index_col=0)
with open("/media/ubuntu/sda/Spike_Sorting/paper_architecture/01_real_data/01_flexible_probe_30_channels/kilosort_spike_sorting/sorting_new/021322/neuron_inf.pkl", 'rb') as f:
    neuron_inf = pickle.load(f)

In [67]:
recording_raw = se.read_blackrock(file_path='/media/ubuntu/sda/data/mouse6/ns4/natural_image/mouse6_021322_natural_image_001.ns4')
recording_recorded = recording_raw.remove_channels(["98", '31', '32'])

recording_f = spre.bandpass_filter(recording_recorded, freq_min=300, freq_max=3000)
recording_f = spre.common_reference(recording_f, reference="global", operator="median")

In [72]:
# ===== AutoSort 的 detect_spike 函数（完全一致） =====
import scipy.signal

def detect_spike(
    trace0_car,
    thr_min=3,
    thr_max=30,
    distance=3,
    ch_max_simul_firing=5,
    wlen=5,
    prominence=10,
):
    """
    AutoSort 的阈值检测函数（与 detection.py 完全一致）
    
    参数:
        trace0_car: numpy数组，形状为 (n_timepoints, n_channels)
        thr_min: 最小阈值倍数（相对于噪声标准差），默认3
        thr_max: 最大阈值倍数（用于过滤异常值），默认30
        distance: 峰值之间的最小距离（采样点），默认3
        ch_max_simul_firing: 同时放电的最大通道数，默认5
        wlen: 峰值检测的窗口长度，默认5
        prominence: 峰值的最小突出度，默认10
    
    返回:
        spikes: 二进制矩阵 (n_timepoints, n_channels)，1表示检测到spike
    """
    noise_std_detect = np.median(abs(trace0_car) / 0.6745, axis=0)
    thr = thr_min * noise_std_detect
    thrmax = thr_max * noise_std_detect

    spikes = np.zeros(trace0_car.shape)
    if trace0_car.ndim > 1:
        for i in range(noise_std_detect.shape[0]):
            peaks, props = scipy.signal.find_peaks(
                -trace0_car[:, i],
                height=thr[i],
                distance=distance,
                wlen=wlen,
                prominence=prominence,
            )
            prominences = scipy.signal.peak_prominences(
                -trace0_car[:, i], peaks, wlen=7
            )[0]
            peaks = peaks[props["peak_heights"] > 10]
            prominences = prominences[props["peak_heights"] > 10]
            peaks = peaks[(prominences > 15)]

            spikes[peaks, i] = 1

        # larger value no more than thrmax
        points = trace0_car.shape[0]
        spike_coord = np.argwhere(spikes == 1)
        for i in range(spike_coord.shape[0]):
            near_start = spike_coord[i, 0] - 5
            near_end = spike_coord[i, 0] + 5
            if near_start < 0:
                near_start = 0
            if near_end >= points:
                near_end = points - 1
            if np.any(np.max(trace0_car[near_start:near_end, :], axis=0) >= thrmax):
                spikes[spike_coord[i, 0], spike_coord[i, 1]] = 0

        # no simultanous firing!!!!
        thres_cross = ch_max_simul_firing
        spikes[np.sum(spikes, axis=1) > thres_cross, :] = 0
    return spikes


# ===== AutoSort 的 map_gt_annotation 函数（向量化优化版本） =====
def map_gt_annotation(detect_array, gt_array):
    """
    AutoSort 的 GT 映射函数（向量化优化版本，逻辑与 detection.py 完全一致）
    
    参数:
        detect_array: numpy数组，形状为 (n_detected, 2)，每行为 [时间点, 通道ID]
        gt_array: numpy数组，形状为 (n_gt, 2)，每行为 [时间点, 通道ID]
    
    返回:
        gt_label_array1: numpy数组，形状为 (n_detected,)，值为对应的 GT 索引或 -1（未匹配）
    """
    n_detected = detect_array.shape[0]
    gt_label_array1 = np.full(n_detected, -1, dtype=np.int64)
    
    if n_detected == 0 or gt_array.shape[0] == 0:
        return gt_label_array1
    
    # 提取检测到的时间和通道
    detect_times = detect_array[:, 0].astype(np.int64)
    detect_channels = detect_array[:, 1].astype(np.int64)
    
    # 提取 GT 的时间和通道
    gt_times = gt_array[:, 0].astype(np.int64)
    gt_channels = gt_array[:, 1].astype(np.int64)
    
    # 使用字典来加速查找：key = (时间, 通道), value = GT索引列表
    # 这样可以O(1)查找，避免重复扫描整个gt_array
    from collections import defaultdict
    gt_dict = defaultdict(list)
    for idx, (t, c) in enumerate(zip(gt_times, gt_channels)):
        gt_dict[(t, c)].append(idx)
    
    # 为每个检测到的 spike 尝试匹配三种时间偏移：0, -1, +1（按优先级）
    time_offsets = [0, -1, 1]
    
    # 向量化匹配：对每个时间偏移，批量处理所有未匹配的检测 spike
    for offset in time_offsets:
        # 找到还未匹配的检测 spike
        unmatched_mask = gt_label_array1 == -1
        if not np.any(unmatched_mask):
            break
        
        # 计算偏移后的时间（只对未匹配的）
        unmatched_indices = np.where(unmatched_mask)[0]
        shifted_times = detect_times[unmatched_indices] + offset
        unmatched_channels = detect_channels[unmatched_indices]
        
        # 向量化查找：使用字典快速匹配（O(1)查找）
        # 使用列表推导式批量构建键，然后批量查找
        keys = [(shifted_times[i], unmatched_channels[i]) for i in range(len(unmatched_indices))]
        
        # 批量查找匹配（避免逐个循环查找）
        for i, key in enumerate(keys):
            if key in gt_dict and len(gt_dict[key]) > 0:
                # 找到匹配，使用第一个匹配的 GT 索引
                gt_idx = gt_dict[key][0]
                detect_idx = unmatched_indices[i]
                gt_label_array1[detect_idx] = gt_idx
                # 从字典中移除已匹配的项（避免重复匹配）
                gt_dict[key].pop(0)
                if len(gt_dict[key]) == 0:
                    del gt_dict[key]
    
    return gt_label_array1
def extract_windows(data, indices, window_size=30):
    """
    根据给定的时间点索引提取窗口
    与train_spike_pipeline.py保持一致：
    - left_sample = 10 (spike前10个采样点)
    - right_sample = 20 (spike后20个采样点)
    - 总共30个采样点: [spike_time - 10, spike_time + 19]
    """
    n_channels, time_length = data.shape
    left_sample = 30   # 与train_spike_pipeline.py保持一致
    right_sample = 30  # 与train_spike_pipeline.py保持一致
    
    # 验证边界
    if np.any(indices < left_sample) or np.any(indices >= time_length - right_sample):
        raise ValueError("Some indices are out of bounds for the given window size.")

    windows = []
    for idx in indices:
        # 提取 [idx - 10, idx + 20)，共30个时间点
        window = data[:, idx - left_sample:idx + right_sample]
        windows.append(window)

    windows = np.array(windows)
    return windows

  

channel_groups = np.sort(neuron_inf['tract_channel'].unique())


In [83]:
print("### 2. detect spikes (使用 AutoSort 的检测逻辑)")

# 获取recording的采样率和通道数
sampling_rate = recording_f.get_sampling_frequency()
n_channels = recording_f.get_num_channels()
print(f"采样率: {sampling_rate} Hz, 通道数: {n_channels}")

# 计算200秒对应的采样点数
duration_seconds = 200
max_frames = int(duration_seconds * sampling_rate)
total_frames = recording_f.get_num_frames()
actual_frames = min(max_frames, total_frames)

print(f"Recording总长度: {total_frames} 采样点 ({total_frames/sampling_rate:.2f} 秒)")
print(f"将处理前 {actual_frames} 采样点 ({actual_frames/sampling_rate:.2f} 秒)")

# 读取数据（与 AutoSort 的 generate_autosort_input 一致）
trace0_car = recording_f.get_traces(start_frame=0, end_frame=actual_frames).astype(np.float32)
# trace0_car 形状: (n_timepoints, n_channels)

print(f"数据形状: {trace0_car.shape}")

# 使用 AutoSort 的 detect_spike 函数（参数与 generate_autosort_input 完全一致）
spikes = detect_spike(
    trace0_car,
    thr_min=3.5,
    thr_max=30,
    distance=3,
    ch_max_simul_firing=5,
    wlen=5,
    prominence=10,
)
# spikes 形状: (n_timepoints, n_channels)，1 表示检测到 spike

# 按照 AutoSort 的方式构建 detect_array
print("构建 detect_array...")
spiketrain = {}
all_spike_train = []
spike_loc = []
for channel_num in range(trace0_car.shape[1]):
    spiketrain_loc = np.where(spikes[:, channel_num])[0]
    spiketrain[channel_num] = spiketrain_loc
    all_spike_train += list(spiketrain_loc)
    spike_loc += [channel_num] * len(spiketrain_loc)

X_spiketrain_time = all_spike_train
Y_spiketrain_id_final = spike_loc
detect_array = np.array([X_spiketrain_time, Y_spiketrain_id_final]).T
# detect_array 形状: (n_detected, 2)，每行为 [时间点, 通道ID]

print(f"检测到的 spike 数量: {len(detect_array)}")
print(f"detect_array 形状: {detect_array.shape}")



### 2. detect spikes (使用 AutoSort 的检测逻辑)
采样率: 10000.0 Hz, 通道数: 30
Recording总长度: 40000100 采样点 (4000.01 秒)
将处理前 2000000 采样点 (200.00 秒)
数据形状: (2000000, 30)
构建 detect_array...
检测到的 spike 数量: 431532
detect_array 形状: (431532, 2)


In [85]:
print("### 3. load ground truth")

# 计算200秒对应的采样点数
if 'sampling_rate' not in locals():
    sampling_rate = recording_f.get_sampling_frequency()
duration_seconds = 200
max_frames = int(duration_seconds * sampling_rate)

# 过滤spike_inf，只保留前200秒的数据
spike_inf_filtered = spike_inf[spike_inf['time'] < max_frames].copy()

# 按照 AutoSort 的方式构建 gt_array
# AutoSort 使用 firings_merged.npz，这里我们使用 spike_inf 和 neuron_inf
print("构建 gt_array...")
spike_train_all = []
y_unit_id = []
gt_ch = []

# 从 spike_inf 和 neuron_inf 构建 GT 数据
for neuron_idx in range(len(neuron_inf)):
    neuron_name = neuron_inf['Neuron'].iloc[neuron_idx]
    neuron_channel_id = neuron_inf['tract_channel'].iloc[neuron_idx]
    
    # 获取该 neuron 的所有 spike 时间
    neuron_spikes = spike_inf_filtered[spike_inf_filtered['neuron'] == neuron_name]
    if len(neuron_spikes) > 0:
        spike_times = neuron_spikes['time'].values
        spike_train_all += list(spike_times)
        y_unit_id += [neuron_name] * len(spike_times)
        gt_ch += [neuron_channel_id] * len(spike_times)

# 构建 gt_array（与 AutoSort 一致）
gt_array = np.array([spike_train_all, gt_ch]).T
# gt_array 形状: (n_gt, 2)，每行为 [时间点, 通道ID]

print(f"GT spike 数量: {len(gt_array)}")
print(f"gt_array 形状: {gt_array.shape}")

print("\n### 4. map ground truth annotation")

# 使用 AutoSort 的 map_gt_annotation 函数
gt_label_array1 = map_gt_annotation(detect_array, gt_array)

# 计算检测率（与 AutoSort 一致）
detection_rate = np.where(gt_label_array1 > -1)[0].shape[0] / gt_array.shape[0]
print(f"---spike detection rate: {detection_rate:.4f}")

# 按照 AutoSort 的方式构建 Y_spiketrain_id
# 使用 object 类型以支持存储字符串（neuron 名称）
Y_spiketrain_id = np.full((detect_array.shape[0],), None, dtype=object)
matched_indices = np.where(gt_label_array1 > -1)[0]
if len(matched_indices) > 0:
    # 将 y_unit_id 转换为 numpy 数组（object 类型以支持字符串）
    y_unit_id_array = np.array(y_unit_id, dtype=object)
    Y_spiketrain_id[matched_indices] = y_unit_id_array[
        gt_label_array1[matched_indices].astype("int")
    ]

print(f"匹配到的 spike 数量: {len(matched_indices)}")
print(f"未匹配的 spike 数量: {len(detect_array) - len(matched_indices)}")

# 打印统计信息
print("\nGT标注映射统计:")
unique_units = np.unique(y_unit_id)
print(f"总 GT unit 数量: {len(unique_units)}")
print(f"总 GT spike 数量: {len(gt_array)}")
print(f"检测到的 spike 数量: {len(detect_array)}")
print(f"匹配到的 spike 数量: {len(matched_indices)}")
print(f"检测率: {detection_rate:.4f}")

# 按 unit 统计匹配情况
print("\n按 unit 统计:")
for unit in unique_units[:10]:  # 只显示前10个
    unit_gt_count = np.sum(np.array(y_unit_id) == unit)
    unit_matched_indices = matched_indices[Y_spiketrain_id[matched_indices] == unit]
    unit_matched_count = len(unit_matched_indices)
    if unit_gt_count > 0:
        unit_detection_rate = unit_matched_count / unit_gt_count
        print(f"  {unit}: GT={unit_gt_count}, 匹配={unit_matched_count}, 检测率={unit_detection_rate:.4f}")

print("\n### 5. find corresponding waveform (按照 AutoSort 的方式)")

# 设置窗口参数（与 AutoSort 一致）
left_sample = 10
right_sample = 20

# 过滤边界附近的 spike（确保可以提取完整的窗口）
# 与 AutoSort 的 generate_autosort_input 完全一致
# 注意：X_spiketrain_time 和 Y_spiketrain_id_final 在 Cell 4 中已定义
X_spiketrain_time = np.array(X_spiketrain_time)
valid_mask = X_spiketrain_time < trace0_car.shape[0] - (left_sample + right_sample)

X_spiketrain_time = X_spiketrain_time[valid_mask]
Y_spiketrain_id = Y_spiketrain_id[valid_mask]
Y_spiketrain_id_final = np.array(Y_spiketrain_id_final)[valid_mask]

print(f"过滤边界后的 spike 数量: {len(X_spiketrain_time)}")
print(f"窗口参数: left_sample={left_sample}, right_sample={right_sample}")
print(f"窗口总长度: {left_sample + right_sample} 采样点")

# 按照 AutoSort 的方式提取窗口
# 使用循环从 -left_sample 到 right_sample-1 提取每个时间点的数据
print("开始提取波形...")
for time_range in tqdm(np.arange(-left_sample, right_sample), desc="提取波形"):
    if time_range == -left_sample:
        # 第一个时间点，初始化 waveform
        waveform = trace0_car[X_spiketrain_time + time_range, :]
    else:
        # 后续时间点，使用 dstack 堆叠
        waveform = np.dstack(
            (waveform, trace0_car[X_spiketrain_time + time_range, :])
        )

# waveform 形状: (n_spikes, n_channels, window_length)
print(f"波形提取完成！")
print(f"waveform 形状: {waveform.shape}")
print(f"  - n_spikes: {waveform.shape[0]}")
print(f"  - n_channels: {waveform.shape[1]}")
print(f"  - window_length: {waveform.shape[2]}")



### 3. load ground truth
构建 gt_array...
GT spike 数量: 74790
gt_array 形状: (74790, 2)

### 4. map ground truth annotation
---spike detection rate: 0.9266
匹配到的 spike 数量: 69300
未匹配的 spike 数量: 362232

GT标注映射统计:
总 GT unit 数量: 27
总 GT spike 数量: 74790
检测到的 spike 数量: 431532
匹配到的 spike 数量: 69300
检测率: 0.9266

按 unit 统计:
  Neuron_1: GT=5262, 匹配=5064, 检测率=0.9624
  Neuron_11: GT=829, 匹配=819, 检测率=0.9879
  Neuron_17: GT=1525, 匹配=1403, 检测率=0.9200
  Neuron_2: GT=2637, 匹配=1597, 检测率=0.6056
  Neuron_20: GT=904, 匹配=885, 检测率=0.9790
  Neuron_22: GT=980, 匹配=918, 检测率=0.9367
  Neuron_24: GT=352, 匹配=301, 检测率=0.8551
  Neuron_25: GT=594, 匹配=547, 检测率=0.9209
  Neuron_27: GT=458, 匹配=416, 检测率=0.9083
  Neuron_28: GT=729, 匹配=670, 检测率=0.9191

### 5. find corresponding waveform (按照 AutoSort 的方式)
过滤边界后的 spike 数量: 431525
窗口参数: left_sample=10, right_sample=20
窗口总长度: 30 采样点
开始提取波形...


提取波形: 100%|██████████| 30/30 [00:07<00:00,  3.82it/s]

波形提取完成！
waveform 形状: (431525, 30, 30)
  - n_spikes: 431525
  - n_channels: 30
  - window_length: 30


In [87]:
print("### 6. 准备训练数据（按照 AutoSort 的方式，但不包含位置信息）")

# 保存数据（与 AutoSort 的格式一致）
import pickle
from pathlib import Path

# 创建保存目录
save_dir = "/media/ubuntu/sda/Spike_Sorting/paper_architecture/01_real_data/01_flexible_probe_30_channels/autosort_input/"
train_data_dir = Path(save_dir) / "train_data"
train_data_dir.mkdir(parents=True, exist_ok=True)

print(f"保存目录: {train_data_dir}")

# 准备数据
# waveform 形状: (n_spikes, n_channels, window_length)
# 需要转换为 AutoSort 的格式

# 1. X_waveform: 多通道波形 (n_spikes, n_channels, window_length)
X_waveform = waveform  # 已经是正确格式
print(f"X_waveform 形状: {X_waveform.shape}")

# 2. Y_spike_id: 单元ID（从 Y_spiketrain_id 转换）
# Y_spiketrain_id 包含 neuron 名称（字符串），需要转换为整数索引
unique_neurons = np.unique([x for x in Y_spiketrain_id if x is not None])
neuron_to_id = {neuron: idx for idx, neuron in enumerate(unique_neurons)}
neuron_to_id[None] = -1  # 噪声标记为 -1

Y_spike_id = np.array([neuron_to_id.get(x, -1) for x in Y_spiketrain_id])
print(f"Y_spike_id 形状: {Y_spike_id.shape}")
print(f"唯一单元数量: {len(unique_neurons)} (不包括噪声)")

# 3. Y_spike_id_noise: 通道ID（用于提取 single waveform）
# 从 Y_spiketrain_id_final 获取（这是检测到的通道ID）
Y_spike_id_noise = Y_spiketrain_id_final
print(f"Y_spike_id_noise 形状: {Y_spike_id_noise.shape}")

# 4. X_spiketrain_time: 时间点
# 已经定义
print(f"X_spiketrain_time 形状: {X_spiketrain_time.shape}")

# 保存数据
print("\n保存数据...")
with open(train_data_dir / "X_waveform.pkl", "wb") as f:
    pickle.dump(X_waveform, f)
print(f"  ✓ X_waveform.pkl 已保存")

with open(train_data_dir / "Y_spike_id.pkl", "wb") as f:
    pickle.dump(Y_spike_id, f)
print(f"  ✓ Y_spike_id.pkl 已保存")

with open(train_data_dir / "Y_spike_id_noise.pkl", "wb") as f:
    pickle.dump(Y_spike_id_noise, f)
print(f"  ✓ Y_spike_id_noise.pkl 已保存")

with open(train_data_dir / "X_spiketrain_time.pkl", "wb") as f:
    pickle.dump(X_spiketrain_time, f)
print(f"  ✓ X_spiketrain_time.pkl 已保存")

print(f"\n所有数据已保存到: {train_data_dir}")
print(f"数据统计:")
print(f"  - 总 spike 数量: {len(X_waveform)}")
print(f"  - 通道数: {X_waveform.shape[1]}")
print(f"  - 窗口长度: {X_waveform.shape[2]}")
print(f"  - 唯一单元数: {len(unique_neurons)}")
print(f"  - 噪声 spike 数量: {np.sum(Y_spike_id == -1)}")
print(f"  - 有效 spike 数量: {np.sum(Y_spike_id != -1)}")


### 6. 准备训练数据（按照 AutoSort 的方式，但不包含位置信息）
保存目录: /media/ubuntu/sda/Spike_Sorting/paper_architecture/01_real_data/01_flexible_probe_30_channels/autosort_input/train_data
X_waveform 形状: (431525, 30, 30)
Y_spike_id 形状: (431525,)
唯一单元数量: 27 (不包括噪声)
Y_spike_id_noise 形状: (431525,)
X_spiketrain_time 形状: (431525,)

保存数据...
  ✓ X_waveform.pkl 已保存
  ✓ Y_spike_id.pkl 已保存
  ✓ Y_spike_id_noise.pkl 已保存
  ✓ X_spiketrain_time.pkl 已保存

所有数据已保存到: /media/ubuntu/sda/Spike_Sorting/paper_architecture/01_real_data/01_flexible_probe_30_channels/autosort_input/train_data
数据统计:
  - 总 spike 数量: 431525
  - 通道数: 30
  - 窗口长度: 30
  - 唯一单元数: 27
  - 噪声 spike 数量: 362227
  - 有效 spike 数量: 69298


In [88]:
print("### 7. 创建 Dataset Loader（不包含位置信息）")

import torch
from torch.utils import data
from torch.utils.data import random_split

class SimpleWaveformLoader(data.Dataset):
    """
    简化的 waveform loader，不包含位置信息
    只使用 multi-waveform 和 single-waveform
    """
    def __init__(self, root, shank_channel, Keep_id=None):
        # 加载数据
        with open(root + "X_waveform.pkl", "rb") as f:
            datafile = pickle.load(f)
        try:
            with open(root + "Y_spike_id.pkl", "rb") as f:
                GT = pickle.load(f)
        except FileNotFoundError:
            GT = np.zeros(datafile.shape[0]) - 1
        with open(root + "Y_spike_id_noise.pkl", "rb") as f:
            channel_id = np.array(pickle.load(f))
        
        # 确定要保留的单元ID
        if Keep_id is None:
            Keep_id = np.unique(GT)
            Keep_id = list(Keep_id[Keep_id != -1])
            self.keep_id = Keep_id
        else:
            self.keep_id = Keep_id
        
        # 创建噪声/非噪声标签
        mask = ~np.isin(GT, Keep_id)
        GT = np.array(GT)
        
        GT_binary = np.zeros((GT.shape[0], 2))
        GT_binary[list(mask), 0] = 1  # 噪声
        GT_binary[~mask, 1] = 1       # 非噪声
        
        self.GT_unique = Keep_id + [-1]
        self.GT_binary = GT_binary
        
        # 提取 single waveform（从最大幅度通道）
        self.Img_single = datafile[np.arange(datafile.shape[0]), np.array(channel_id).astype('int'), :]
        
        self.GT_LIST = GT
        
        # 创建单元分类标签（one-hot）
        GT_array = np.zeros((len(GT), len(Keep_id)))
        for idx, unique_id in enumerate(Keep_id):
            rmv_list = np.where(np.array(GT) == unique_id)[0]
            GT_array[rmv_list, idx] = 1
        self.GT = GT_array
        
        self.Img = datafile  # 多通道波形
        
        # 计算类别权重（用于处理不平衡数据）
        self.pos_weight_noise = torch.tensor([
            -np.sum(self.GT_binary[:,0]-1)/np.sum(self.GT_binary[:,0]),
            -np.sum(self.GT_binary[:,1]-1)/np.sum(self.GT_binary[:,1])
        ])
        self.pos_weight_label = torch.tensor([
            -(np.sum(self.GT[:,i]-1)+sum(np.sum(GT_array,axis=1)==0))/np.sum(self.GT[:,i]) 
            for i in range(self.GT.shape[1])
        ])
        
        self.n_classes = len(set(self.GT_unique))
        
        print(f"Dataset 加载完成:")
        print(f"  - 总样本数: {len(self.GT)}")
        print(f"  - 通道数: {self.Img.shape[1]}")
        print(f"  - 窗口长度: {self.Img.shape[2]}")
        print(f"  - 唯一单元数: {len(Keep_id)}")
        print(f"  - 噪声样本数: {np.sum(self.GT_binary[:, 0])}")
        print(f"  - 非噪声样本数: {np.sum(self.GT_binary[:, 1])}")
    
    def __len__(self):
        return len(self.GT)
    
    def __getitem__(self, index):
        # 返回: 多通道波形, 单元分类标签, 噪声/非噪声标签, 单通道波形
        # 注意：不返回位置信息
        return (
            self.Img[index, ...],      # (n_channels, window_length)
            self.GT[index, ...],       # (n_units,) one-hot
            self.GT_binary[index, ...], # (2,) [noise, spike]
            self.Img_single[index, ...] # (window_length,)
        )

# 测试 dataset
print("创建 dataset...")
dataset = SimpleWaveformLoader(
    root=str(train_data_dir) + '/',
    shank_channel=np.arange(n_channels),
    Keep_id=None
)

print(f"\nDataset 创建成功！")
print(f"keep_id (单元ID列表): {dataset.keep_id}")


### 7. 创建 Dataset Loader（不包含位置信息）
创建 dataset...
Dataset 加载完成:
  - 总样本数: 431525
  - 通道数: 30
  - 窗口长度: 30
  - 唯一单元数: 27
  - 噪声样本数: 362227.0
  - 非噪声样本数: 69298.0

Dataset 创建成功！
keep_id (单元ID列表): [np.int64(0), np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7), np.int64(8), np.int64(9), np.int64(10), np.int64(11), np.int64(12), np.int64(13), np.int64(14), np.int64(15), np.int64(16), np.int64(17), np.int64(18), np.int64(19), np.int64(20), np.int64(21), np.int64(22), np.int64(23), np.int64(24), np.int64(25), np.int64(26)]


In [ ]:
print("### 8. 创建简化的 AutoSort Model（不包含位置信息）")

import torch.nn as nn
import torch.nn.functional as nnf

class SimpleClassifier(nn.Module):
    """
    简化的分类器（与 AutoSort 的 clssimp 相同）
    """
    def __init__(self, input_dim, num_classes):
        super(SimpleClassifier, self).__init__()
        self.pool = nn.AdaptiveAvgPool1d(output_size=(input_dim))
        self.way1 = nn.Sequential(
            nn.Linear(input_dim, 1000, bias=True),
            nn.BatchNorm1d(1000),
            nn.ReLU(inplace=True),
        )
        self.way2 = nn.Sequential(
            nn.Linear(1000, 512, bias=True),
            nn.BatchNorm1d(512),
            nn.ReLU(inplace=True),
        )
        self.way3 = nn.Sequential(
            nn.Linear(512, 100, bias=True),
            nn.BatchNorm1d(100),
            nn.ReLU(inplace=True),
        )
        self.cls = nn.Linear(100, num_classes, bias=True)

    def forward(self, x):
        x = self.pool(x[None, :])
        x = x.reshape(x.size(1), -1)
        x = self.way1(x)
        x = self.way2(x)
        x = self.way3(x)
        logits = self.cls(x)
        return logits

    def intermediate_forward(self, x):
        x = self.pool(x[None, :])
        x = x.reshape(x.size(1), -1)
        x = self.way1(x)
        x = self.way2(x)
        x = self.way3(x)
        return x


class SimpleAutoSort:
    """
    简化的 AutoSort 模型（不包含位置信息）
    输入: multi-waveform + single-waveform
    与原始 AutoSort 完全一致，只是去掉了位置信息
    """
    def __init__(self, ch_num, samplepoints, device, set_shank_id, save_dir, 
                 pos_weight_noise=None, pos_weight_label=None):
        # 输入维度: (ch_num + 1) * samplepoints（不包含位置信息）
        # 原始: (ch_num + 1) * samplepoints + loc_dim
        input_dim = (ch_num + 1) * samplepoints
        
        self.clsfier_noise = SimpleClassifier(input_dim, 2).to(device)
        self.clsfier_label = SimpleClassifier(input_dim, len(set_shank_id)).to(device)
        
        self.optimizer = torch.optim.Adam([
            {'params': self.clsfier_noise.parameters()},
            {'params': self.clsfier_label.parameters()},
        ], lr=1e-4)
        
        self.criterion = nn.MSELoss()  # 与原始一致（虽然不使用）
        self.bceloss = nn.BCEWithLogitsLoss(pos_weight=pos_weight_noise)
        self.bceloss_label = nn.BCEWithLogitsLoss(pos_weight=pos_weight_label)
        
        # 与原始 AutoSort 的保存路径命名一致（只是去掉位置信息）
        self.save_model_path_1 = save_dir + 'multitask_single_wave_noise_ae.pth'  # 不使用，但保留
        self.save_model_path_2 = save_dir + 'multitask_single_wave_clsfier_noise_clsfier.pth'
        self.save_model_path_3 = save_dir + 'multitask_single_wave_clsfier_label_clsfier.pth'
        
        self.set_shank_id = set_shank_id
        self.device = device
    
    def save_model(self):
        # 与原始 AutoSort 完全一致
        torch.save(self.clsfier_noise.state_dict(), self.save_model_path_2)
        torch.save(self.clsfier_label.state_dict(), self.save_model_path_3)
    
    def load_model(self):
        # 与原始 AutoSort 完全一致
        self.clsfier_noise.load_state_dict(torch.load(self.save_model_path_2))
        self.clsfier_label.load_state_dict(torch.load(self.save_model_path_3))
    
    def train(self):
        self.clsfier_noise.train()
        self.clsfier_label.train()
    
    def eval(self):
        self.clsfier_noise.eval()
        self.clsfier_label.eval()
    
    def iter_model(self, batch_features, classify_labels, labels, single_waveform):
        """
        训练迭代
        输入:
            batch_features: (batch_size, ch_num * samplepoints) - 多通道波形展平
            classify_labels: (batch_size, n_units) - 单元分类标签
            labels: (batch_size, 2) - 噪声/非噪声标签
            single_waveform: (batch_size, samplepoints) - 单通道波形
        """
        self.optimizer.zero_grad()
        
        # 拼接 multi-waveform 和 single-waveform
        codes = torch.cat((batch_features, single_waveform), axis=1)
        # codes 形状: (batch_size, (ch_num+1)*samplepoints)
        
        # 噪声分类
        cls_output = self.clsfier_noise(codes.float())
        
        # 单元分类（只对非噪声样本）
        test = labels[:, 1] == 1
        if sum(test) > 1:
            cls_label_output = self.clsfier_label(codes.float()[test, :])
            train_loss3 = 1000 * self.bceloss_label(
                cls_label_output, 
                classify_labels[test, :len(self.set_shank_id)]
            )
        else:
            train_loss3 = torch.tensor(0)
        
        train_loss2 = 1000 * self.bceloss(cls_output, labels)
        
        train_loss = train_loss2 + train_loss3
        train_loss.backward()
        self.optimizer.step()
        
        return train_loss2.item(), train_loss3.item(), test
    
    def iter_model_eval(self, batch_features, classify_labels, labels, single_waveform):
        """
        评估迭代
        """
        codes = torch.cat((batch_features, single_waveform), axis=1)
        
        cls_output = self.clsfier_noise(codes.float())
        gt = torch.argmax(labels, axis=1)
        pred = torch.argmax(cls_output, axis=1)
        
        test = labels[:, 1] == 1
        if sum(test) > 1:
            cls_label_output = self.clsfier_label(codes.float()[test, :])
            pred_class = torch.argmax(cls_label_output, axis=1)
            gt_label_class = torch.argmax(classify_labels[test, :len(self.set_shank_id)], axis=1)
            train_loss3 = 1000 * self.bceloss_label(
                cls_label_output, 
                classify_labels[test, :len(self.set_shank_id)]
            )
        else:
            train_loss3 = torch.tensor(0)
            gt_label_class = torch.tensor([])
            pred_class = torch.tensor([])
        
        train_loss2 = 1000 * self.bceloss(cls_output, labels)
        train_loss = train_loss2 + train_loss3
        
        return train_loss2.item(), train_loss3.item(), gt, pred, gt_label_class, pred_class

print("Model 类定义完成！")


### 8. 创建简化的 AutoSort Model（不包含位置信息）
Model 类定义完成！


In [ ]:
print("### 9. 训练 AutoSort Model")

from sklearn.metrics import accuracy_score, f1_score

# 设置设备
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"使用设备: {device}")

# 创建模型保存目录
model_save_dir = save_dir + "model_save/"
Path(model_save_dir).mkdir(parents=True, exist_ok=True)

# 设置参数
ch_num = n_channels
samplepoints = left_sample + right_sample  # 30
set_shank_id = dataset.keep_id

print(f"模型参数:")
print(f"  - 通道数: {ch_num}")
print(f"  - 窗口长度: {samplepoints}")
print(f"  - 单元数量: {len(set_shank_id)}")
print(f"  - 输入维度: {(ch_num + 1) * samplepoints}")

# 创建模型
autosort_model = SimpleAutoSort(
    ch_num=ch_num,
    samplepoints=samplepoints,
    device=device,
    set_shank_id=set_shank_id,
    save_dir=model_save_dir,
    pos_weight_noise=dataset.pos_weight_noise.to(device),
    pos_weight_label=dataset.pos_weight_label.to(device)
)

# 划分训练集和验证集
train_size = int(len(dataset) * 0.8)
val_size = len(dataset) - train_size
train_dataset, val_dataset = random_split(dataset, [train_size, val_size])

train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=512, shuffle=True)
val_loader = torch.utils.data.DataLoader(val_dataset, batch_size=512, shuffle=False)

print(f"\n数据集划分:")
print(f"  - 训练集: {train_size} 样本")
print(f"  - 验证集: {val_size} 样本")

# 训练参数（与原始 AutoSort 完全一致）
epochs = 20
min_valid_loss = np.inf

# 检查模型是否已存在
import os
if os.path.exists(autosort_model.save_model_path_2):
    autosort_model.load_model()
    autosort_model.to_device(device)
    print("已加载现有模型")
else:
    training_log = {'epoch': [],
                    'validation_acc_noise':[],
                    'validation_acc_label':[]}

    print(f"\n开始训练（共 {epochs} 个 epoch）...")

    for epoch in range(epochs):
        training_log['epoch'].append(epoch + 1)
        print("epoch : {}/{}".format(epoch + 1, epochs))

        loss2 = 0
        loss3 = 0
        autosort_model.train()
        autosort_model.bceloss.pos_weight = autosort_model.bceloss.pos_weight.to(device)
        autosort_model.bceloss_label.pos_weight = autosort_model.bceloss_label.pos_weight.to(device)
        
        for batch_features, classify_labels, labels, single_waveform in tqdm(train_loader, desc="训练"):
            classify_labels = classify_labels.to(device)
            batch_features = batch_features.view(-1, samplepoints * ch_num).to(device)
            labels = labels.to(device)
            single_waveform = single_waveform.to(device)

            train_loss2, train_loss3, test = autosort_model.iter_model(
                batch_features, classify_labels, labels, single_waveform
            )

            loss2 += train_loss2
            if sum(test) > 0:
                loss3 += train_loss3

        loss2 = loss2 / len(train_loader)
        loss3 = loss3 / len(train_loader)
        print("epoch : {}/{}, detection loss = {:.6f}, classification loss = {:.6f}".format(
            epoch + 1, epochs, loss2, loss3))

        valid_loss2 = 0.0
        valid_loss3 = 0.0

        gt_all = []
        pred_all = []
        gt_class_all = []
        pred_class_all = []
        autosort_model.eval()
        
        for batch_features, classify_labels, labels, single_waveform in tqdm(val_loader, desc="验证"):
            classify_labels = classify_labels.to(device)
            batch_features = batch_features.view(-1, samplepoints * ch_num).to(device)
            labels = labels.to(device)
            single_waveform = single_waveform.to(device)

            valid_loss2_batch, valid_loss3_batch, gt, pred, gt_label_class, pred_class = autosort_model.iter_model_eval(
                batch_features, classify_labels, labels, single_waveform
            )

            valid_loss2 += valid_loss2_batch
            valid_loss3 += valid_loss3_batch

            gt_all.append(gt.detach().cpu().numpy())
            pred_all.append(pred.detach().cpu().numpy())
            # 与原始代码一致：即使 gt_label_class 是空 tensor，也追加（空数组）
            pred_class_all.append(pred_class.detach().cpu().numpy())
            gt_class_all.append(gt_label_class.detach().cpu().numpy())

        gt_all = np.concatenate(gt_all, axis=0)
        pred_all = np.concatenate(pred_all, axis=0)
        # 过滤空数组（与原始代码兼容）
        gt_class_all = [x for x in gt_class_all if len(x) > 0]
        pred_class_all = [x for x in pred_class_all if len(x) > 0]
        if len(gt_class_all) > 0:
            gt_class_all = np.concatenate(gt_class_all, axis=0)
            pred_class_all = np.concatenate(pred_class_all, axis=0)
        else:
            gt_class_all = np.array([])
            pred_class_all = np.array([])

        valid_loss2 = valid_loss2 / len(val_loader)
        valid_loss3 = valid_loss3 / len(val_loader)
        valid_loss =  valid_loss2 + valid_loss3
        print("epoch : {}/{}, detection loss = {:.6f}, classification loss = {:.6f}".format(
            epoch + 1, epochs, valid_loss2, valid_loss3))

        training_log['validation_acc_noise'].append(accuracy_score(gt_all, pred_all))
        # 与原始代码一致：使用 f1_score with average='micro'
        if len(gt_class_all) > 0:
            training_log['validation_acc_label'].append(f1_score(gt_class_all, pred_class_all, average='micro'))
        else:
            training_log['validation_acc_label'].append(0.0)

        if min_valid_loss > valid_loss:
            print(f'Validation Loss Decreased({min_valid_loss:.6f}--->{valid_loss:.6f}) \t Saving The Model')
            min_valid_loss = valid_loss
            # Saving State Dict
            autosort_model.save_model()

    # 保存训练日志（与原始 AutoSort 完全一致）
    pd.DataFrame(training_log).to_csv(model_save_dir + 'training_log.csv')
    print(f"\n训练完成！训练日志已保存到: {model_save_dir}training_log.csv")


### 9. 训练 AutoSort Model
使用设备: cuda
模型参数:
  - 通道数: 30
  - 窗口长度: 30
  - 单元数量: 27
  - 输入维度: 930

数据集划分:
  - 训练集: 345220 样本
  - 验证集: 86305 样本

开始训练（共 20 个 epoch）...
epoch : 1/20


训练: 100%|██████████| 675/675 [00:05<00:00, 120.71it/s]


epoch : 1/20, detection loss = 0.000000, classification loss = 365.711007


验证: 100%|██████████| 169/169 [00:00<00:00, 200.34it/s]


epoch : 1/20, detection loss = 0.000000, classification loss = 308.953005
Validation Loss Decreased(inf--->693.487850) 	 Saving The Model
epoch : 2/20


...

epoch : 19/20, detection loss = 0.000000, classification loss = 601.618321
epoch : 20/20


训练: 100%|██████████| 675/675 [00:05<00:00, 129.72it/s]


epoch : 20/20, detection loss = 0.000000, classification loss = 26.107731


验证: 100%|██████████| 169/169 [00:00<00:00, 194.78it/s]

epoch : 20/20, detection loss = 0.000000, classification loss = 529.510483

训练完成！训练日志已保存到: /media/ubuntu/sda/Spike_Sorting/paper_architecture/01_real_data/01_flexible_probe_30_channels/autosort_input/model_save/training_log.csv
